# Phase 6: Conditional Diffusion Downscaling

This notebook demonstrates synthetic statistical downscaling from approximately 12 km to approximately 5 km precipitation fields. Generated samples are implementation-validation outputs, not calibrated meteorological probabilities or operational forecasts.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import torch

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.downscaling.synthetic import generate_paired_synthetic_dataset
from src.downscaling.diffusion_dataset import DiffusionDownscalingDataset
from src.downscaling.diffusion import ConditionalDiffusionUNet, DiffusionLoss, DiffusionProcess
from src.downscaling.diffusion_sampler import generate, summarize_ensemble

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
samples = generate_paired_synthetic_dataset(num_events=2, timesteps_per_event=2, seed=42)
dataset = DiffusionDownscalingDataset(samples)
item = dataset[0]
condition = item['condition'].unsqueeze(0).to(device)
target = item['target'].unsqueeze(0).to(device)
print(condition.shape, target.shape, item['event_id'], item['track_id'])

## Forward diffusion
The target is transformed with `log1p`, then Gaussian noise is added at selected timesteps.

In [ ]:
process = DiffusionProcess(timesteps=20, beta_schedule='cosine').to(device)
noise = torch.randn_like(target)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for axis, step in zip(axes, [0, 5, 10, 19]):
    timestep = torch.tensor([step], device=device)
    noisy = process.q_sample(target, timestep, noise)
    axis.imshow(noisy[0, 0].detach().cpu(), cmap='Blues')
    axis.set_title(f't={step}')
    axis.axis('off')
plt.tight_layout()

## Conditional model and tiny training run

In [ ]:
model = ConditionalDiffusionUNet(base_channels=8).to(device)
criterion = DiffusionLoss(extreme_enabled=True, extreme_percentile=95.0, extreme_weight=2.0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.002)
losses = []
for _ in range(8):
    timestep = torch.randint(0, process.timesteps, (1,), device=device)
    noise = torch.randn_like(target)
    noisy = process.q_sample(target, timestep, noise)
    loss = criterion(model(noisy, timestep, condition), noise, target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach()))
print(f'loss: {losses[0]:.4f} -> {losses[-1]:.4f}')

In [ ]:
generated = generate(model, process, condition, num_samples=5, seed=42)
ensemble = summarize_ensemble(generated)
fig, axes = plt.subplots(1, 4, figsize=(14, 3))
for axis, (title, field) in zip(axes, [('Condition', condition), ('Sample 1', generated[:, 0]), ('Ensemble mean', ensemble['mean']), ('Ensemble std', ensemble['std'])]):
    axis.imshow(field[0, 0].detach().cpu(), cmap='Blues')
    axis.set_title(title)
    axis.axis('off')
plt.tight_layout()

## Scientific interpretation
The ensemble spread represents model-generated sample variability. This notebook does not establish calibrated meteorological probabilities, physical consistency, or operational forecasting skill.